# Mocks

## A quoi servent les mocks
Les mocks sont des objets simulés qui imitent le comportement d'objets réels dans un environnement contrôlé. Ils sont principalement utilisés dans les tests unitaires pour isoler les unités de code et vérifier leur comportement sans dépendre de ressources externes (comme des bases de données, des API ou des services réseau). Les mocks permettent de :

 * Isoler le code testé : En simulant les dépendances, on peut tester une unité de code sans interférences.
 * Contrôler les réponses : On peut spécifier les valeurs de retour des méthodes et les exceptions à lever.
 * Vérifier les interactions : On peut s'assurer que certaines méthodes sont appelées avec les bons arguments.

## patch

### Principe de base

Le principe de mock et plusparticuliérement de patch est de remplacer a la volée un objet python pour en controler le résultat.
Il est possible de le faire à la main.


In [1]:
import random

# La fonction originale
def get_random_number():
    return random.randint(0, 10)

# Le mock qu'on utilise
def mock_randint( a, b ):
    return 5

# Contrôle du résultat
def simulate_random_number_manually():
    # Sauvegarder la référence originale
    original_randint = random.randint
    # Remplacer random.randint par une fonction qui retourne un nombre fixe
    random.randint = mock_randint # Forcer le retour à 5
    # Appeler la fonction qui utilise random.randint
    result = get_random_number()
    # Restauration de la fonction originale
    random.randint = original_randint
    return result

if __name__ == "__main__":
    for _ in range( 5 ):
        print(f"Le nombre aléatoire simulé sans patch est : {simulate_random_number_manually()}")
        print(f"Le nombre aléatoire : {get_random_number()}" ) 


Le nombre aléatoire simulé sans patch est : 5
Le nombre aléatoire : 2
Le nombre aléatoire simulé sans patch est : 5
Le nombre aléatoire : 5
Le nombre aléatoire simulé sans patch est : 5
Le nombre aléatoire : 7
Le nombre aléatoire simulé sans patch est : 5
Le nombre aléatoire : 0
Le nombre aléatoire simulé sans patch est : 5
Le nombre aléatoire : 1


### En utilisant le module mock d'unittest

Le module mock de unittest offre les outils pour patcher une méthode 

In [2]:
import random
from unittest import mock


# La fonction originale
def get_random_number():
    return random.randint(0, 10)

# Le mock qu'on utilise
def mock_randint( a, b ):
    return 5

# Contrôle du résultat avec patch
def simulate_random_number_patched():
    # On définit le chemin vers la méthode randint
    randint_path = "random.randint"
    # On créé un mock pour patcher la méthode
    with mock.patch( randint_path ) as mck_randint:
        # On modifie le retour de randint
        mck_randint.side_effect = mock_randint
        # Appeler la fonction qui utilise random.randint
        result = get_random_number()
    return result


if __name__ == "__main__":
    for _ in range( 5 ):
        print(f"Le nombre aléatoire simulé avec patch est : {simulate_random_number_patched()}")
        print(f"Le nombre aléatoire : {get_random_number()}" ) 


Le nombre aléatoire simulé avec patch est : 5
Le nombre aléatoire : 4
Le nombre aléatoire simulé avec patch est : 5
Le nombre aléatoire : 5
Le nombre aléatoire simulé avec patch est : 5
Le nombre aléatoire : 8
Le nombre aléatoire simulé avec patch est : 5
Le nombre aléatoire : 2
Le nombre aléatoire simulé avec patch est : 5
Le nombre aléatoire : 0


## La simulation d'objet 

Les modules unittest.mock de Python fournissent des outils puissants pour créer des objets simulés (ou "mocks") qui imitent le comportement d'objets réels. Deux des classes les plus couramment utilisées dans ce module sont Mock et MagicMock. Ces classes permettent de simuler des objets et de contrôler leurs comportements.

[Documentation](https://docs.python.org/3/library/unittest.mock.html)

### Mock

La classe Mock est une classe de base pour créer des objets simulés. Elle permet de créer des objets qui peuvent remplacer des dépendances dans votre code et d'enregistrer comment ils sont utilisés.

#### Caractéristiques de Mock :
 * **Création d'objets simulés** : Vous pouvez créer des instances de Mock pour simuler des objets.
 * **Définition de comportements** : Vous pouvez définir des valeurs de retour pour les méthodes de l'objet simulé.
 * **Vérification des interactions** : Vous pouvez vérifier si certaines méthodes ont été appelées, combien de fois, et avec quels arguments.
   
#### Exemple d'utilisation de Mock :

In [3]:
from unittest.mock import Mock

# Création d'un mock
mock_object = Mock()

# Définition du comportement
mock_object.some_method.return_value = 'mocked value'

# Appel de la méthode
result = mock_object.some_method()

# Vérification du résultat
print(result)  # Affiche : mocked value

# Vérification des appels
mock_object.some_method.assert_called_once()


mocked value


### MagicMock
MagicMock est une sous-classe de Mock qui est spécialement conçue pour gérer les dunder méthodes. Les dunder méthodes sont les méthodes spéciales dans Python qui commencent et se terminent par des doubles underscores, comme `__getitem__`, `__len__`, `__iter__`, etc. Ces méthodes permettent de définir des comportements spéciaux pour les objets en Python.

#### Caractéristiques de MagicMock :
 * **Support des méthodes magiques** : MagicMock permet de simuler facilement les méthodes magiques, ce qui le rend utile lorsque vous devez tester des objets qui se comportent comme des conteneurs ou des itérables.
 * **Fonctionnalité similaire à Mock** : Il possède toutes les fonctionnalités de Mock, tout en ajoutant la prise en charge des méthodes magiques.

#### Exemple d'utilisation de MagicMock :


In [4]:
from unittest.mock import MagicMock

# Création d'un MagicMock
magic_mock_object = MagicMock()

# Définition du comportement pour une méthode magique
magic_mock_object.__len__.return_value = 3

# Vérification de la longueur
length = len(magic_mock_object)

# Affichage du résultat
print(length)  # Affiche : 3

# Vérification des appels
magic_mock_object.__len__.assert_called_once()


3



## Integration dans les tests unitaires

### Le premier example avec la fixture monkeypatch

Code d'example dans /example/mock_random/.

Votre module `module.py` contenant le code a tester 

```python
import random

# La fonction originale
def get_random_number():
    return random.randint(0, 10)
```

Votre fichier de test `test_random.py`

```python
import pytest

def test_get_random_number(monkeypatch):
    # Il est trés important d'importer random
    from module import get_random_number, random

    # Utiliser monkeypatch pour remplacer random.randint
    monkeypatch.setattr(random, 'randint', lambda a, b: 5)  # Simuler le retour de 5

    # Appeler la fonction à tester
    result = get_random_number()

    # Vérifier que le résultat est celui attendu
    assert result == 5
```

### Example decorateur patch

La méthode patch de la librairie unittest peut être utilisée comme décorateur



In [5]:
import random
from unittest.mock import Mock, patch


# La fonction originale
def get_random_number():
    return random.randint(0, 10)

# Contrôle du résultat avec le decorateur patch
@patch('random.randint', Mock(return_value=4))
def simulate_random_number_decorator_patched():
    result = get_random_number()
    return result


if __name__ == "__main__":
    for _ in range( 5 ):
        print(f"Le nombre aléatoire simulé avec patch est : {simulate_random_number_decorator_patched()}")
        print(f"Le nombre aléatoire : {get_random_number()}" ) 

Le nombre aléatoire simulé avec patch est : 4
Le nombre aléatoire : 7
Le nombre aléatoire simulé avec patch est : 4
Le nombre aléatoire : 1
Le nombre aléatoire simulé avec patch est : 4
Le nombre aléatoire : 5
Le nombre aléatoire simulé avec patch est : 4
Le nombre aléatoire : 2
Le nombre aléatoire simulé avec patch est : 4
Le nombre aléatoire : 9


## Examples utiles

Quelques éxamples que j'ai pu mettre en place

### Mock env

On peut surcharger le dictionnaire os.environ avec la méthode `mock.patch.dict`

#### fixture

Un exemple pour l'intégrer à une fixture pytest. Le mot clef `yield` ici est trés important car il permet de ne pas sortir du with.

```python
import pytest
from unittest import mock
import os


@pytest.fixture()
def mock_vars():
    """
    fixture to mock environment variables
    """
    values = {
        "TEST_VARS": "test",
    }
    with mock.patch.dict(os.environ, values):
        yield


def test_vars(mock_vars):

    assert os.environ["TEST_VARS"] == "test"
```

### Mock requests

Le package `requests-mock` permet de créer des mocks sur le package `requests` de python.  
[Documentation](https://requests-mock.readthedocs.io/en/latest/)

#### fixture

On peut simplifier l'usage en l'intégrant a une fixture, comme l'example précédent le `yield` est trés important car il permet de rester dans le with.


```python
import pytest
import requests_mock
import requests


@pytest.fixture()
def mock_url():
    """
    fixture to mock a url
    """
    with requests_mock.Mocker() as m:
        m.get( "https://fake_url/test", json={ "result": "ok" })
        yield


def test_url(mock_url):
    assert requests.get( "https://fake_url/test" ).json() == {"result": "ok"}
```